# Código do Luís (Ollama) a correr no Colab

Corre o `run_inference_LangChain_HF.py` **sem alterações**: instala o Ollama dentro da máquina do Colab, descarrega o
mesmo modelo e usa o script como está. Antes: **Runtime → Change runtime type → GPU (T4)**, senão o Ollama usa o
processador e fica lento.

In [ ]:
# Branch publicada no GitHub
REPO_URL = "https://github.com/Luis-FO/Prompt-Engineering-for-KE.git"
BRANCH = "rui/colab"
MODEL = "qwen3.5:4b"   # o mesmo modelo do README

import os
os.chdir("/content")
if not os.path.isdir("kl"):
    !git clone -b {BRANCH} {REPO_URL} kl
os.chdir("/content/kl")
!git log --oneline -1

Cloning into 'kl'...
remote: Enumerating objects: 752, done.
remote: Counting objects: 100% (344/344), done.
remote: Compressing objects: 100% (177/177), done.
remote: Total 752 (delta 225), reused 250 (delta 163), pack-reused 408 (from 1)
Receiving objects: 100% (752/752), 40.60 MiB | 19.38 MiB/s, done.
Resolving deltas: 100% (537/537), done.
Updating files: 100% (459/459), done.
a4e2db0 (HEAD -> rui/colab, origin/rui/colab) Notebook Colab com Ollama


In [10]:
# Instalar o Ollama na máquina do Colab, arrancar o servidor e descarregar o modelo
!apt-get install -y -q zstd
!curl -fsSL https://ollama.com/install.sh | sh

import subprocess, time, urllib.request
subprocess.Popen(["ollama", "serve"], stdout=open("/content/ollama.log", "w"), stderr=subprocess.STDOUT)
for _ in range(60):
    try:
        urllib.request.urlopen("http://localhost:11434"); break
    except Exception:
        time.sleep(1)
else:
    raise RuntimeError("O servidor do Ollama não arrancou. Vê /content/ollama.log")
!ollama pull {MODEL}

Reading package lists...
Building dependency tree...
Reading state information...
zstd is already the newest version (1.5.5+dfsg2-2build1.1).
0 upgraded, 0 newly installed, 0 to remove and 33 not upgraded.
>>> Cleaning up old version at /usr/local/lib/ollama
>>> Installing ollama to /usr/local
>>> Downloading ollama-linux-amd64.tar.zst
######################################################################## 100.0%
>>> Adding ollama user to video group...
>>> Adding current user to ollama group...
>>> Creating ollama systemd service...
>>> The Ollama API is now available at 127.0.0.1:11434.
>>> Install complete. Run "ollama" from the command line.



In [ ]:
# Dependências do script
!pip install -q langchain-ollama tqdm retry

## 1. Teste com 3 prompts


In [12]:
import json
SRC = "1_data_module/2_creating_prompts/data_with_prompts/three_shot_RAG_prompts_test.json"
os.makedirs("/content/prompts_teste", exist_ok=True)
json.dump(json.load(open(SRC))[:3], open("/content/prompts_teste/three_shot_RAG_prompts_test.json", "w"))
!python 2_inference_module/run_inference_LangChain_HF.py --input_folder /content/prompts_teste --model_name {MODEL} --out_dir /content/out_teste

INFO:root:Start Logging
INFO:root:Read the input file: /content/prompts_teste/three_shot_RAG_prompts_test.json. Number of instances: 3. Extraction started.
Passing inputs through qwen3.5:4b for inference:   0% 0/3 [00:00<?, ?it/s]INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Passing inputs through qwen3.5:4b for inference:  33% 1/3 [02:41<05:23, 161.78s/it]INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Passing inputs through qwen3.5:4b for inference:  67% 2/3 [02:43<01:07, 67.62s/it] INFO:httpx:HTTP Request: POST http://127.0.0.1:11434/api/chat "HTTP/1.1 200 OK"
Passing inputs through qwen3.5:4b for inference: 100% 3/3 [02:45<00:00, 55.07s/it]
INFO:root:Extraction finished. Saved the results to /content/out_teste/three_shot_RAG_prompts_test.json
Elapsed time: 2.76 minutes


In [13]:
for x in json.load(open("/content/out_teste/three_shot_RAG_prompts_test.json")):
    print(repr(x["generated_text"][:300]))

"[['Porsche Panamera', 'Manufacturer', 'Porsche'], ['Porsche Panamera', 'Engine layout', 'Front-engined'], ['Porsche Panamera', 'Drive layout', 'Rear-wheel-drive']]"
"[['Paramount+', 'Owned by', 'ViacomCBS Streaming'], ['ViacomCBS Streaming', 'Division of', 'ViacomCBS'], ['Paramount+', 'Offers content from', 'CBS broadcast properties'], ['Paramount+', 'Offers content from', 'ViacomCBS library']]"
"['Paradox', 'Created', 'board games'], ['Paradox', 'Owned rights to', 'World of Darkness'], ['Paradox', 'Purchased White Wolf Publishing', '2015'], ['Paradox', 'Hosts annual convention', 'PDXCon'], ['PDXCon', 'Open to public since', '2017']"


## 2. Corrida completa (446 prompts, só o three_shot_RAG)
O script só grava no fim de cada ficheiro. Se o Colab desligar a meio, perde-se esta run. Guardamos o resultado no Drive.

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
OUT = "/content/drive/MyDrive/kl_out_ollama"

os.makedirs("/content/prompts_rag", exist_ok=True)
!cp {SRC} /content/prompts_rag/
!python 2_inference_module/run_inference_LangChain_HF.py --input_folder /content/prompts_rag --model_name {MODEL} --out_dir {OUT}

## 3. Post-processamento e avaliação

In [ ]:
!python 2_inference_module/post_process_generated_string.py --input_folder {OUT} --out_dir {OUT}_pp
!python 3_evaluation_module/evaluation_reference_to_annotations/evaluate.py --input_dir {OUT}_pp --gold_file 1_data_module/1_data_preprocessing/RED-fm/test.jsonl --out_dir {OUT}_eval